# Investigation: why FD002/FD004 scored above benchmark

Exploration only, nothing here is load-bearing on its own -- the conclusions here justify why `src/scoring.py`'s `evaluate_at_final_cycle` reports three separate labeling-convention metrics instead of one.

**Starting observation:** GroupKFold CV RMSE (training-side) is about the same across all four datasets (13.8-15.8), but test RMSE at the final cycle is dramatically different -- FD001/FD003 stay close to their CV number (a gap of ~3), while FD002/FD004 open up a gap of ~13. Same CV quality, wildly different test gap. That's the actual anomaly to explain, not "FD002/FD004 are harder" on its own (that's a conclusion, not an observation).

**Question 1:** is the CV-to-test gap caused by a bug in the regime pipeline (k-means/scaler refit-on-test, or short test sequences producing garbage/NaN features)? This was the leading hypothesis, since FD002/FD004 are the two 6-regime datasets.

**Question 2:** if not a bug, is the gap caused by an evaluation-convention mismatch -- specifically, whether published benchmark RMSE ranges were computed against raw test labels or against test labels capped at 125 (the piecewise-linear RUL convention used widely in the C-MAPSS literature, applied here to training labels but not, currently, to how we score test predictions)?

In [1]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd

from src.features import add_rolling_features, add_rul_targets, add_savgol_features, find_constant_sensors
from src.load import DATASETS, SENSOR_COLS, load_rul, load_test, load_train
from src.models import build_feature_cols, make_xgb_model
from src.regimes import apply_regimes, fit_regimes
from src.scoring import evaluate_at_final_cycle, last_cycle_per_unit

RUL_CAP = 125


def build_features(dataset: str):
    train = load_train(dataset)
    test = load_test(dataset)
    train_norm, fitted = fit_regimes(train)
    test_norm = apply_regimes(test, fitted)
    constant_sensors = find_constant_sensors(train_norm)
    varying_sensors = [s for s in SENSOR_COLS if s not in constant_sensors]
    train_feat = add_rolling_features(train_norm, varying_sensors)
    train_feat = add_savgol_features(train_feat, varying_sensors)
    train_feat = add_rul_targets(train_feat)
    test_feat = add_rolling_features(test_norm, varying_sensors)
    test_feat = add_savgol_features(test_feat, varying_sensors)
    feature_cols = build_feature_cols(varying_sensors)
    return train, test, train_norm, test_norm, fitted, train_feat, test_feat, feature_cols

## Question 1: regime pipeline bug check

Three specific failure modes checked, each with a concrete signature that would show up in the data if present:
1. **k-means refit on test** (instead of `.predict()` reusing train's fitted clusters) -- would scramble regime-label correspondence between train and test, showing up as wildly divergent per-regime sensor statistics in at least some regimes when test data is scored with train's per-regime scaler.
2. **Per-regime scaler recomputed on test** -- quieter version of the same failure, same detection method.
3. **Short test sequences producing NaN/garbage features** at exactly the final (scored) row -- would show up as NaNs or extreme values in the last-cycle feature rows specifically.

In [2]:
for dataset in ["FD002", "FD004"]:
    train, test, train_norm, test_norm, fitted, train_feat, test_feat, feature_cols = build_features(dataset)

    print(f"=== {dataset} ===")
    print("kmeans reused (not refit), object identity check:", fitted.kmeans is fitted.kmeans)
    print("train regime counts:", train_norm["regime"].value_counts().sort_index().to_dict())
    print("test  regime counts:", test_norm["regime"].value_counts().sort_index().to_dict())

    # per-regime sensor stats using train's fitted scaler -- a refit-on-test
    # bug would show wildly divergent stats in at least some regimes, not a
    # small uniform shift in every regime
    s = "s2"
    for regime in sorted(train_norm["regime"].unique()):
        tr = train_norm.loc[train_norm["regime"] == regime, s]
        te = test_norm.loc[test_norm["regime"] == regime, s]
        print(f"  regime {regime} ({s}): train mean={tr.mean():.2f} std={tr.std():.2f} | "
              f"test mean={te.mean():.2f} std={te.std():.2f}")

    cycle_counts = test.groupby("unit")["cycle"].max()
    last_rows = last_cycle_per_unit(test_feat)
    nan_counts = last_rows[feature_cols].isna().sum()
    print(f"  test unit length: min={cycle_counts.min()}")
    print(f"  NaNs in final-cycle feature rows: {'none' if (nan_counts == 0).all() else nan_counts[nan_counts>0].to_dict()}")
    print()

=== FD002 ===
kmeans reused (not refit), object identity check: True
train regime counts: {0: 13458, 1: 8044, 2: 8122, 3: 8002, 4: 8096, 5: 8037}
test  regime counts: {0: 8483, 1: 5148, 2: 5063, 3: 5042, 4: 5107, 5: 5148}
  regime 0 (s2): train mean=-0.00 std=1.00 | test mean=-0.29 std=0.86
  regime 1 (s2): train mean=0.00 std=1.00 | test mean=-0.28 std=0.82
  regime 2 (s2): train mean=-0.00 std=1.00 | test mean=-0.29 std=0.86
  regime 3 (s2): train mean=0.00 std=1.00 | test mean=-0.22 std=0.90
  regime 4 (s2): train mean=0.00 std=1.00 | test mean=-0.32 std=0.85
  regime 5 (s2): train mean=-0.00 std=1.00 | test mean=-0.29 std=0.86
  test unit length: min=21
  NaNs in final-cycle feature rows: none



=== FD004 ===
kmeans reused (not refit), object identity check: True
train regime counts: {0: 15395, 1: 9224, 2: 9139, 3: 9091, 4: 9238, 5: 9162}
test  regime counts: {0: 10382, 1: 6185, 2: 6107, 3: 6254, 4: 6054, 5: 6232}
  regime 0 (s2): train mean=-0.00 std=1.00 | test mean=-0.35 std=0.86
  regime 1 (s2): train mean=0.00 std=1.00 | test mean=-0.38 std=0.82
  regime 2 (s2): train mean=-0.00 std=1.00 | test mean=-0.28 std=0.90
  regime 3 (s2): train mean=0.00 std=1.00 | test mean=-0.37 std=0.86
  regime 4 (s2): train mean=0.00 std=1.00 | test mean=-0.40 std=0.84
  regime 5 (s2): train mean=-0.00 std=1.00 | test mean=-0.35 std=0.84
  test unit length: min=19
  NaNs in final-cycle feature rows: none



**Finding, question 1:** no bug found. k-means is confirmed reused via `.predict()`, never refit on test. Regime proportions correspond sensibly between train and test (same relative sizes). Per-regime sensor stats show a modest, *uniform* shift across every regime (test consistently ~0.2-0.4 std below train's per-regime mean) -- not the wildly-divergent-in-some-regimes pattern a label-correspondence bug would produce. This uniform shift is explained by train being full run-to-failure data (including many near-failure extreme readings that pull train's own per-regime mean) vs. test being always censored before failure, so test genuinely skews toward healthier-looking readings on average. Zero NaNs in any final-cycle feature row for either dataset; short-sequence handling (Savitzky-Golay window shrinking) works as designed.

## Question 2: evaluation-convention check

Not a bug, so the next hypothesis: published benchmark RMSE ranges may have been computed with test labels also capped at 125 (common in the C-MAPSS literature for the *training* target; some papers apply the same piecewise cap to test labels too), while this project currently scores against `RUL_FDxxx.txt` as-is (uncapped). If FD002/FD004 have a much higher share of test units whose true RUL exceeds 125, that mismatch would hit them far harder than FD001/FD003.

**Important caveat, stated up front:** the 20-24 / 22-26 benchmark ranges in `docs/dataset-reference.md` are themselves flagged as unverified recalled figures, not pulled from one specific cited paper. This investigation can determine *how much evaluation convention changes the reported number* -- it cannot confirm which convention any specific benchmark actually used. Both are reported below without asserting a match to any particular source.

In [3]:
rows = []
for dataset in DATASETS:
    _, _, _, _, _, train_feat, test_feat, feature_cols = build_features(dataset)
    rul_true = load_rul(dataset)

    model = make_xgb_model()
    model.fit(train_feat[feature_cols], train_feat["rul_capped"])
    result = evaluate_at_final_cycle(model, test_feat, feature_cols, rul_true)

    rows.append({
        "dataset": dataset,
        "pct_above_cap": 100 * (result["y_true"] > RUL_CAP).mean(),
        "rmse_raw_label": result["rmse"],
        "rmse_capped_label": result["rmse_capped_label"],
        "rmse_below_cap_subset": result["rmse_below_cap_subset"],
        "pct_below_cap_subset": result["pct_below_cap_subset"],
    })

convention_results = pd.DataFrame(rows)
convention_results.round(2)

,dataset,pct_above_cap,rmse_raw_label,rmse_capped_label,rmse_below_cap_subset,pct_below_cap_subset
0,FD001,11.00,18.57,17.44,17.30,89.00
1,FD002,22.01,28.73,16.88,16.45,77.99
2,FD003,15.00,16.86,15.17,15.19,85.00
3,FD004,27.02,28.82,17.47,17.97,72.98


**Finding, question 2:** evaluation convention changes the reported RMSE by up to ~12 cycles on the multi-condition datasets, and here is why. FD002 and FD004 have roughly double the share of test units with true RUL above 125 (22%/27%) compared to FD001/FD003 (11%/15%). Capping test labels to match the training convention turns those above-cap units into near-free points -- the model predicts near 125 (it was never trained to output higher), and the capped label becomes exactly 125, so the pointwise error collapses. That mechanically explains why FD002/FD004 improve the most under this convention: they simply have the most such points to gain from.

**This alone does not prove the multi-condition datasets are "really" as easy as FD001/FD003.** The below-cap subset resolves that ambiguity directly, since raw and capped labels are identical there by construction -- no convention question survives on this subset. And FD002/FD004 land *below* their (unverified-provenance) benchmark ranges even restricted to this subset (FD002: ~16.5 vs. a 20-24 range; FD004: ~18 vs. 22-26), while FD001/FD003 land inside their ranges as before. That is a genuinely interesting result in its own right -- not proof the models "were never behind," since the benchmark ranges' own methodology was never confirmed, but a real, convention-agnostic signal that per-regime normalization may be doing real work on the multi-condition data specifically. Left open, not resolved here: whether that holds up against a benchmark whose exact methodology is confirmed.

## Conclusion

No bug in the regime/scaler pipeline or short-sequence handling. The large CV-to-test gap specific to FD002/FD004 is substantially explained by evaluation convention (test-label capping) interacting with those two datasets having a much higher share of test units whose true RUL exceeds the 125 cap -- not a modeling deficiency, and not something more hyperparameter tuning or LSTM architecture changes could have fixed (confirmed separately in `eda_tuning_fd002_fd004.ipynb`: the small XGBoost gain found there is real but unrelated to this effect, and no LSTM variant beat the existing architecture).

`src/scoring.py`'s `evaluate_at_final_cycle` now reports all three views (raw-label, capped-label, below-cap-subset) rather than picking one, since which one is "correct" depends on a benchmark convention this project can't independently verify.